<a href="https://colab.research.google.com/github/sahithyanjali/Sahithyanjali/blob/main/Day15_Enterprise_Banking_Decision_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 15 — Enterprise Banking Decision Architecture

**Instructor:** Uma Desu  
**Program:** GenAI Pioneer Summer Training  

This notebook demonstrates a simple 3-layer enterprise banking decision engine for transaction governance.

## Concept

A real bank rarely trusts one AI model directly. Decision systems combine deterministic rules, anomaly scores, compliance checks, audit lineage, and human escalation paths.

In [7]:
import json
import time
import collections # Needed for deque
from datetime import datetime # Added for parsing timestamps

# --- MOCK INBOUND TRANSACTION DATA STREAM ---
# Added 'timestamp' and 'account_tier' for adaptive threshold testing
TRANSACTIONS = [
    {
        "tx_id": "TX-99102",
        "user_id": "USR-4412",
        "amount": 45000,          # Within normal balance limits
        "location": "Hyderabad",
        "device": "iPhone14",
        "memo": "Monthly grocery payment to local vendor",
        "timestamp": "2024-07-26T14:30:00Z",
        "account_tier": "VIP"
    },
    {
        "tx_id": "TX-99103",
        "user_id": "USR-8821",
        "amount": 7500000,         # Massive spike vs historical average
        "location": "São Paulo",   # Geographic velocity anomaly
        "device": "Unknown_Linux_Box",
        "memo": "Urgent crypto asset settlement to offshore beneficiary shell co",
        "timestamp": "2024-07-26T15:00:00Z",
        "account_tier": "STANDARD"
    },
    # --- Transactions for Velocity Filter Testing ---
    {
        "tx_id": "TX-V001",
        "user_id": "USR-VELOCITY",
        "amount": 100,
        "location": "New York",
        "device": "iPhone",
        "memo": "Velocity test transaction 1",
        "timestamp": "2024-07-26T10:00:00Z",
        "account_tier": "STANDARD"
    },
    {
        "tx_id": "TX-V002",
        "user_id": "USR-VELOCITY",
        "amount": 100,
        "location": "New York",
        "device": "iPhone",
        "memo": "Velocity test transaction 2",
        "timestamp": "2024-07-26T10:00:00Z",
        "account_tier": "STANDARD"
    },
    {
        "tx_id": "TX-V003",
        "user_id": "USR-VELOCITY",
        "amount": 100,
        "location": "New York",
        "device": "iPhone",
        "memo": "Velocity test transaction 3",
        "timestamp": "2024-07-26T10:00:00Z",
        "account_tier": "STANDARD"
    },
    {
        "tx_id": "TX-V004", # This transaction should trigger the velocity throttle
        "user_id": "USR-VELOCITY",
        "amount": 100,
        "location": "New York",
        "device": "iPhone",
        "memo": "Velocity test transaction 4 (should be blocked)",
        "timestamp": "2024-07-26T10:00:00Z",
        "account_tier": "STANDARD"
    },
    # --- Transactions for Adaptive Threshold Testing (new) ---
    {
        "tx_id": "TX-AT-001", # VIP, high amount, should use relaxed threshold
        "user_id": "USR-4412", # VIP user
        "amount": 250000,
        "location": "London",
        "device": "iPhone14",
        "memo": "Large investment transfer",
        "timestamp": "2024-07-26T11:00:00Z",
        "account_tier": "VIP"
    },
    {
        "tx_id": "TX-AT-002", # Standard, high-risk hours, should use conservative threshold
        "user_id": "USR-8821",
        "amount": 25000,
        "location": "Berlin",
        "device": "Android_S23",
        "memo": "Late night online purchase",
        "timestamp": "2024-07-26T03:00:00Z", # 3 AM - high risk
        "account_tier": "STANDARD"
    },
    {
        "tx_id": "TX-AT-003", # VIP, very high amount, should pass relaxed threshold
        "user_id": "USR-4412",
        "amount": 500000,
        "location": "Dubai",
        "device": "iPhone14",
        "memo": "Luxury asset acquisition",
        "timestamp": "2024-07-26T13:00:00Z",
        "account_tier": "VIP"
    },
    {
        "tx_id": "TX-AT-004", # Standard, high amount, high-risk hours, should trigger conservative threshold
        "user_id": "USR-8821",
        "amount": 250000,
        "location": "Tokyo",
        "device": "Android_S23",
        "memo": "Early morning stock purchase",
        "timestamp": "2024-07-26T03:30:00Z", # 3:30 AM - high risk
        "account_tier": "STANDARD"
    },
    # --- Transactions from output to ensure consistency ---
    {
        "tx_id": "TX-MR-001",
        "user_id": "USR-8821",
        "amount": 250000,
        "location": "Paris",
        "device": "Android_S23",
        "memo": "Purchase of rare artifact",
        "timestamp": "2024-07-26T16:00:00Z",
        "account_tier": "STANDARD"
    },
    {
        "tx_id": "TX-MR-002",
        "user_id": "USR-4412",
        "amount": 300000,
        "location": "Miami",
        "device": "iPhone14",
        "memo": "Investment transfer",
        "timestamp": "2024-07-26T17:00:00Z",
        "account_tier": "VIP"
    }
]

# --- THE 3-LAYER DECISION ENGINE ---
class EnterpriseDecisionEngine:
    def __init__(self):
        # Layer 1 Data: Mock historical user profiles
        self.user_profiles = {
            "USR-4412": {"avg_tx": 50000, "trusted_devices": ["iPhone14"], "account_tier": "VIP"},
            "USR-8821": {"avg_tx": 25000, "trusted_devices": ["Android_S23"], "account_tier": "STANDARD"},
            "USR-VELOCITY": {"avg_tx": 100, "trusted_devices": ["iPhone"], "account_tier": "STANDARD"} # Profile for velocity test user
        }
        # Layer 3 Data: Regulatory Sanction Keywords (AML compliance)
        self.aml_blacklist = ["shell co", "offshore beneficiary", "sanctioned"]

        # New: Velocity store for sliding window (user_id -> deque of timestamps)
        self.user_velocity_store = collections.defaultdict(collections.deque)
        self.velocity_threshold_seconds = 5 # Window size
        self.velocity_max_transactions = 3  # Max transactions allowed within the window

        # Base anomaly detection threshold
        self.base_anomaly_threshold = 5.0

    def process_transaction(self, tx):
        print(f"\n[INGESTING] Processing Transaction {tx['tx_id']} | Amount: INR {tx['amount']}...")
        start_time = time.time()
        current_tx_time = time.time()

        # Initialize audit_trail with the exact transaction inputs
        audit_trail = {"transaction_inputs": tx.copy()}
        decision = "APPROVE" # Default optimistic decision

        user_id = tx["user_id"]

        # Get user profile or default if not found
        profile = self.user_profiles.get(user_id, {"avg_tx": 10000, "trusted_devices": [], "account_tier": "STANDARD"})

        # --- ADAPTIVE THRESHOLD MATRIX ---
        tx_timestamp_str = tx.get("timestamp", "2024-01-01T00:00:00Z")
        tx_dt = datetime.fromisoformat(tx_timestamp_str.replace("Z", "+00:00"))
        transaction_hour = tx_dt.hour
        account_tier = profile.get("account_tier", "STANDARD")

        effective_threshold = self.base_anomaly_threshold
        threshold_reason = "Default threshold"

        is_new_user = user_id not in self.user_profiles

        # Apply VIP rule first (relaxed threshold)
        if account_tier == "VIP":
            effective_threshold = 6.0 # Relaxed threshold for VIPs
            threshold_reason = "VIP Account Tier (relaxed threshold)"

        # Apply high-risk hours / new user rule (conservative threshold) - this should override VIP if it applies
        if (transaction_hour >= 0 and transaction_hour <= 5) or is_new_user: # 12 AM to 5 AM inclusive
            effective_threshold = 0.85 # Conservative threshold for high-risk times or new users
            threshold_reason = "High-Risk Hours / New User (conservative threshold)"

        audit_trail["Adaptive_Threshold_Decision"] = {
            "transaction_hour": transaction_hour,
            "account_tier": account_tier,
            "is_new_user": is_new_user,
            "base_anomaly_threshold": self.base_anomaly_threshold,
            "effective_threshold_applied": f"{effective_threshold}x deviation",
            "reason": threshold_reason
        }

        # --- VELOCITY FILTER (PRE-LAYER 1 CHECK) ---
        user_tx_timestamps = self.user_velocity_store[user_id]
        velocity_filter_status = "PASS"
        velocity_filter_details = {}

        # Remove transactions older than the defined window
        while user_tx_timestamps and user_tx_timestamps[0] < current_tx_time - self.velocity_threshold_seconds:
            user_tx_timestamps.popleft()

        # Check if velocity threshold is exceeded *before* adding the current transaction
        if len(user_tx_timestamps) >= self.velocity_max_transactions:
            velocity_filter_status = "FAIL"
            velocity_filter_details = {
                "reason": f"Velocity Throttling Triggered (>{self.velocity_max_transactions-1} txs in {self.velocity_threshold_seconds}s)",
                "current_transaction_count": len(user_tx_timestamps) + 1 # Include the current transaction attempt
            }
            decision = "BLOCK_AND_ESCALATE" # Immediately block and escalate
        else:
            user_tx_timestamps.append(current_tx_time)
            velocity_filter_details = {"current_transaction_count": len(user_tx_timestamps)}

        audit_trail["Velocity_Filter_Decision"] = {"status": velocity_filter_status, **velocity_filter_details}

        # Only proceed to other layers if not already blocked by velocity filter
        if decision == "APPROVE":
            # LAYER 1: DETERMINISTIC RULES ENGINE (Sub-millisecond)
            is_trusted_device = tx["device"] in profile["trusted_devices"]
            audit_trail["Layer_1_Rules_Decision"] = {
                "check": "Trusted Device",
                "device_used": tx["device"],
                "trusted_devices_for_user": profile["trusted_devices"],
                "status": "PASS" if is_trusted_device else "FAIL",
                "reason": "" if is_trusted_device else "Untrusted Device Detected"
            }
            if not is_trusted_device:
                decision = "FLAGGED"

            # LAYER 2: MACHINE LEARNING ANOMALY DETECTION (Deterministic score modeling)
            deviation_multiplier = tx["amount"] / profile["avg_tx"]
            anomaly_status = "PASS"
            anomaly_reason = ""

            if deviation_multiplier > effective_threshold: # Use adaptive threshold here
                anomaly_status = "FAIL"
                anomaly_reason = f"Value anomaly score high ({deviation_multiplier:.1f}x deviation > {effective_threshold}x threshold)"
                if decision != "BLOCK_AND_ESCALATE":
                    decision = "FLAGGED"

            audit_trail["Layer_2_ML_Decision"] = {
                "check": "Value Anomaly Detection",
                "transaction_amount": tx["amount"],
                "user_average_transaction": profile["avg_tx"],
                "deviation_multiplier": f"{deviation_multiplier:.2f}x",
                "effective_anomaly_threshold": f"{effective_threshold}x",
                "status": anomaly_status,
                "reason": anomaly_reason
            }

            # LAYER 3: COGNITIVE AML COMPLIANCE LAYER (Text parsing logic)
            aml_trigger = any(keyword in tx["memo"].lower() for keyword in self.aml_blacklist)
            matched_keywords = [keyword for keyword in self.aml_blacklist if keyword in tx["memo"].lower()]
            aml_status = "PASS"
            aml_reason = ""

            if aml_trigger:
                aml_status = "FAIL"
                aml_reason = "Sanctioned text pattern matched in transaction metadata"
                decision = "BLOCK_AND_ESCALATE" # This is the highest severity, always overrides

            audit_trail["Layer_3_AML_Decision"] = {
                "check": "AML Compliance - Memo Scan",
                "memo_text": tx["memo"],
                "aml_blacklist_keywords": self.aml_blacklist,
                "matched_keywords": matched_keywords,
                "status": aml_status,
                "reason": aml_reason
            }

        # --- CENTRAL CONSENSUS MATRIX ---
        # Evaluate aggregated results to determine final systemic routing
        final_verdict = ""
        if decision == "BLOCK_AND_ESCALATE":
            final_verdict = "REJECTED_AND_ROUTED_TO_HUMAN_REVIEW"
        elif decision == "FLAGGED":
            final_verdict = "SUSPENDED_STEP_UP_AUTH_REQUIRED"
        else: # decision == "APPROVE"
            final_verdict = "COMPLETED_SUCCESSFULLY"

        execution_latency_ms = (time.time() - start_time) * 1000

        # Build the final governance output payload
        governance_manifest = {
            "transaction_id": tx["tx_id"],
            "timestamp": tx_timestamp_str, # Use the actual transaction timestamp
            "verdict": final_verdict,
            "latency_ms": f"{execution_latency_ms:.4f}ms",
            "audit_lineage": audit_trail
        }
        return governance_manifest

# --- EXECUTE THE SIMULATION ---
engine = EnterpriseDecisionEngine()
for tx in TRANSACTIONS:
    # Introduce a small delay to simulate real-world transaction arrival over time.
    # This is crucial for the sliding window to work as expected and observe throttling.
    time.sleep(0.1)
    manifest = engine.process_transaction(tx)
    print(json.dumps(manifest, indent=2))
    print("-" * 60)


[INGESTING] Processing Transaction TX-99102 | Amount: INR 45000...
{
  "transaction_id": "TX-99102",
  "timestamp": "2024-07-26T14:30:00Z",
  "verdict": "COMPLETED_SUCCESSFULLY",
  "latency_ms": "0.0904ms",
  "audit_lineage": {
    "transaction_inputs": {
      "tx_id": "TX-99102",
      "user_id": "USR-4412",
      "amount": 45000,
      "location": "Hyderabad",
      "device": "iPhone14",
      "memo": "Monthly grocery payment to local vendor",
      "timestamp": "2024-07-26T14:30:00Z",
      "account_tier": "VIP"
    },
    "Adaptive_Threshold_Decision": {
      "transaction_hour": 14,
      "account_tier": "VIP",
      "is_new_user": false,
      "base_anomaly_threshold": 5.0,
      "effective_threshold_applied": "6.0x deviation",
      "reason": "VIP Account Tier (relaxed threshold)"
    },
    "Velocity_Filter_Decision": {
      "status": "PASS",
      "current_transaction_count": 1
    },
    "Layer_1_Rules_Decision": {
      "check": "Trusted Device",
      "device_used": "iPh

## Teaching Notes

Use this notebook to explain layered decision intelligence:

1. **Rules Layer:** trusted device and structural checks.
2. **Anomaly Layer:** deviation from historical transaction behavior.
3. **AML Layer:** suspicious memo text / compliance triggers.
4. **Consensus Matrix:** final routing decision.
5. **Governance Manifest:** audit-ready decision payload.